# Atividade Prática 04 — Operações com Conjuntos Fuzzy

**Disciplina:** Inteligência Artificial — IFC Araquari

Conjuntos fuzzy discretos. Cada questão tem a **descrição/definição em texto**, a **resposta esperada** (feita na mão) e o **cálculo em Python** validando o resultado com `assert`.

Dados:

$$A = \{0.2/a,\ 0.4/b,\ 1/c,\ 0.8/d,\ 0/e\}$$
$$B = \{0/a,\ 0.9/b,\ 0.3/c,\ 0.2/d,\ 0.1/e\}$$

## Definição dos conjuntos e funções auxiliares

Represento cada conjunto fuzzy como um dicionário `elemento -> grau de pertinência`. As funções abaixo implementam as operações vistas na Unidade 03/04.

In [ ]:
# Conjuntos fuzzy (elemento: grau de pertinencia)
A = {'a': 0.2, 'b': 0.4, 'c': 1.0, 'd': 0.8, 'e': 0.0}
B = {'a': 0.0, 'b': 0.9, 'c': 0.3, 'd': 0.2, 'e': 0.1}

def fmt(S):
    """Formata um conjunto fuzzy no estilo {grau/elemento, ...}."""
    return '{' + ', '.join(f'{round(m,4)}/{x}' for x, m in S.items()) + '}'

# --- Operacoes ---
def suporte(S):        return {x for x, m in S.items() if m > 0}        # mu(x) > 0
def nucleo(S):         return {x for x, m in S.items() if m == 1}       # mu(x) = 1
def cardinalidade(S):  return round(sum(S.values()), 4)                 # soma dos graus
def complemento(S):    return {x: round(1 - m, 4) for x, m in S.items()}        # 1 - mu(x)
def uniao(X, Y):       return {x: max(X[x], Y[x]) for x in X}           # max
def intersecao(X, Y):  return {x: min(X[x], Y[x]) for x in X}           # min
def potencia(S, a):    return {x: round(m ** a, 4) for x, m in S.items()}       # mu(x)^a
def escalar(a, S):     return {x: round(a * m, 4) for x, m in S.items()}        # a * mu(x)
def alpha_corte(S, alpha): return {x for x, m in S.items() if m >= alpha}       # mu(x) >= alpha

print('A =', fmt(A))
print('B =', fmt(B))

A = {0.2/a, 0.4/b, 1.0/c, 0.8/d, 0.0/e}
B = {0.0/a, 0.9/b, 0.3/c, 0.2/d, 0.1/e}


## 1. Suporte

O **suporte** é o conjunto (crisp) dos elementos com grau de pertinência maior que zero:

$$sup(A) = \{x \mid \mu_A(x) > 0\}$$

Resposta esperada:
- `sup(A) = {a, b, c, d}` — o *e* tem μ=0, fica de fora
- `sup(B) = {b, c, d, e}` — o *a* tem μ=0, fica de fora

In [ ]:
print('sup(A) =', sorted(suporte(A)))
print('sup(B) =', sorted(suporte(B)))

assert suporte(A) == {'a','b','c','d'}
assert suporte(B) == {'b','c','d','e'}
print('OK')

sup(A) = ['a', 'b', 'c', 'd']
sup(B) = ['b', 'c', 'd', 'e']
OK


## 2. Núcleo

O **núcleo** é o conjunto dos elementos com pertinência total (igual a 1):

$$nuc(A) = \{x \mid \mu_A(x) = 1\}$$

Resposta esperada:
- `nuc(A) = {c}`
- `nuc(B) = ∅` — nenhum elemento de B chega a 1

In [ ]:
print('nuc(A) =', sorted(nucleo(A)))
print('nuc(B) =', sorted(nucleo(B)), '(vazio)' if not nucleo(B) else '')

assert nucleo(A) == {'c'}
assert nucleo(B) == set()
print('OK')

nuc(A) = ['c']
nuc(B) = [] (vazio)
OK


## 3. Cardinalidade (contagem sigma)

A **cardinalidade** é a soma dos graus de pertinência:

$$card(A) = \sum_i \mu_A(x_i)$$

Resposta esperada:
- `card(A) = 0.2 + 0.4 + 1 + 0.8 + 0 = 2.4`
- `card(B) = 0 + 0.9 + 0.3 + 0.2 + 0.1 = 1.5`

In [ ]:
print('card(A) =', cardinalidade(A))
print('card(B) =', cardinalidade(B))

assert cardinalidade(A) == 2.4
assert cardinalidade(B) == 1.5
print('OK')

card(A) = 2.4
card(B) = 1.5
OK


## 4. Complemento

O **complemento** inverte cada grau de pertinência:

$$\mu_{\neg A}(x) = 1 - \mu_A(x)$$

Resposta esperada:
- `¬A = {0.8/a, 0.6/b, 0/c, 0.2/d, 1/e}`
- `¬B = {1/a, 0.1/b, 0.7/c, 0.8/d, 0.9/e}`

In [ ]:
print('¬A =', fmt(complemento(A)))
print('¬B =', fmt(complemento(B)))

assert complemento(A) == {'a':0.8,'b':0.6,'c':0.0,'d':0.2,'e':1.0}
assert complemento(B) == {'a':1.0,'b':0.1,'c':0.7,'d':0.8,'e':0.9}
print('OK')

¬A = {0.8/a, 0.6/b, 0.0/c, 0.2/d, 1.0/e}
¬B = {1.0/a, 0.1/b, 0.7/c, 0.8/d, 0.9/e}
OK


## 5. União e Interseção

$$\mu_{A\cup B}(x) = \max[\mu_A(x),\ \mu_B(x)] \qquad \mu_{A\cap B}(x) = \min[\mu_A(x),\ \mu_B(x)]$$

Resposta esperada:
- `A ∪ B = {0.2/a, 0.9/b, 1/c, 0.8/d, 0.1/e}`  (pega o **maior** grau de cada elemento)
- `A ∩ B = {0/a, 0.4/b, 0.3/c, 0.2/d, 0/e}`  (pega o **menor** grau de cada elemento)

In [ ]:
print('A ∪ B =', fmt(uniao(A, B)))
print('A ∩ B =', fmt(intersecao(A, B)))

assert uniao(A, B)     == {'a':0.2,'b':0.9,'c':1.0,'d':0.8,'e':0.1}
assert intersecao(A, B) == {'a':0.0,'b':0.4,'c':0.3,'d':0.2,'e':0.0}
print('OK')

A ∪ B = {0.2/a, 0.9/b, 1.0/c, 0.8/d, 0.1/e}
A ∩ B = {0.0/a, 0.4/b, 0.3/c, 0.2/d, 0.0/e}
OK


## 6. Conjunto C = A²  (potência)

Eleva cada grau de pertinência ao expoente *a* (aqui *a* = 2). Isso **concentra** o conjunto (diminui os graus intermediários):

$$A^a = \{\ \mu_A(x)^a \mid \forall x \in X\ \}$$

Resposta esperada:
- `C = A² = {0.04/a, 0.16/b, 1/c, 0.64/d, 0/e}`

In [ ]:
C = potencia(A, 2)
print('C = A² =', fmt(C))

assert C == {'a':0.04,'b':0.16,'c':1.0,'d':0.64,'e':0.0}
print('OK')

C = A² = {0.04/a, 0.16/b, 1.0/c, 0.64/d, 0.0/e}
OK


## 7. Conjunto D = 0.5 × B  (produto escalar)

Multiplica cada grau de pertinência pelo escalar *a* (aqui *a* = 0.5). Isso **dilui** o conjunto:

$$aA = \{\ a\,\mu_A(x) \mid \forall x \in X\ \}$$

Resposta esperada:
- `D = 0.5 × B = {0/a, 0.45/b, 0.15/c, 0.1/d, 0.05/e}`

In [ ]:
D = escalar(0.5, B)
print('D = 0.5 × B =', fmt(D))

assert D == {'a':0.0,'b':0.45,'c':0.15,'d':0.1,'e':0.05}
print('OK')

D = 0.5 × B = {0.0/a, 0.45/b, 0.15/c, 0.1/d, 0.05/e}
OK


## 8. Conjunto E = A₀.₅  (α-corte)

O **α-corte** devolve um conjunto **crisp** com os elementos cujo grau é maior ou igual a α (aqui α = 0.5):

$$A_\alpha = \{x \mid \mu_A(x) \geq \alpha\}$$

Resposta esperada:
- `E = A₀.₅ = {c, d}`  — só *c* (1.0) e *d* (0.8) têm μ ≥ 0.5

In [ ]:
E = alpha_corte(A, 0.5)
print('E = A₀.₅ =', sorted(E))

assert E == {'c','d'}
print('OK')

E = A₀.₅ = ['c', 'd']
OK


---
Todos os `assert` passaram → os cálculos em Python batem com os resultados feitos na mão. ✅